# M5 — Training Strategy & Experimental Design
**DR-Triage · Diabetic Retinopathy Stage Detection**

Covers rubric item 5 (*Training Strategy & Experimental Design*). Everything lives
in `src/train.py`; this notebook (a) proves the pipeline is correct with a fast
**smoke test** you can run right here on a laptop, then (b) launches the
**full report-quality run**, which needs a GPU (Kaggle/Colab).

**Strategy:**

| Choice | What / why |
|---|---|
| Two-phase fine-tuning | Phase 1: frozen backbone, heads only (fast, protects pretrained weights from random-head noise). Phase 2: unfreeze, fine-tune everything. |
| Discriminative learning rates | `lr_backbone` (1e-5) << `lr_head` (1e-3) in phase 2 — the pretrained backbone needs small nudges, the random heads need to learn fast. |
| Progressive resizing | Phase 1 at 224px, phase 2 at 380px — cheaper early epochs, sharper final fine-tune. |
| Cosine annealing w/ warm restarts | Escapes shallow local minima during the long phase-2 fine-tune. |
| Mixed precision (AMP) | On CUDA only — ~2x faster, same accuracy. A no-op on CPU/MPS. |
| MixUp / CutMix | Applied per batch (train split only), p=0.5 — from M3. |
| **3-fold stratified CV** | Reports val QWK as mean ± std, not a single lucky split. |
| **Checkpoint on val QWK** | Not accuracy — QWK is the order-aware, clinically meaningful metric. |
| Early stopping | Patience 6 epochs with no val-QWK improvement. |
| Test-time augmentation | 4 deterministic views averaged at inference (`predict_tta`). |

In [ ]:
# --- Setup ---------------------------------------------------------------------
import sys, os, glob, shutil
from pathlib import Path

PROJECT = None
for cand in ["dr-triage", ".", "..", "/kaggle/working/dr-triage"]:
    if (Path(cand) / "src" / "data.py").is_file():
        PROJECT = str(Path(cand).resolve()); break
if PROJECT is None:
    hits = glob.glob("/kaggle/input/**/src/data.py", recursive=True)
    if hits:
        src_root = str(Path(hits[0]).parents[1]); PROJECT = "/kaggle/working/dr-triage"
        shutil.copytree(src_root, PROJECT, dirs_exist_ok=True)
assert PROJECT, "Upload dr-triage.zip via Add Input -> Upload, then re-run."
os.chdir(PROJECT); sys.path.insert(0, PROJECT)

import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch

from src.data import load_config, load_splits, load_labels
from src.model import DRModel
from src.train import get_device, build_optimizer, cross_validate

cfg = load_config()
device = get_device()
print("device:", device, "| CUDA:", torch.cuda.is_available(), "| MPS:", torch.backends.mps.is_available())

try:
    train_df = load_splits(cfg)["train"]
except FileNotFoundError:
    train_df = load_labels(cfg)
print("train rows:", len(train_df))

if device != "cuda":
    print("\nNo CUDA GPU here — this notebook will run the SMOKE TEST (a few minutes).\n"
          "Run the full training cell (5.3) on Kaggle/Colab with a GPU for report results.")

## 5.1  Discriminative learning-rate groups

A quick look at what `build_optimizer` actually sets up in phase 2: two parameter
groups with different learning rates.

In [ ]:
probe = DRModel(cfg)
opt = build_optimizer(probe, cfg["train"]["lr_backbone"], cfg["train"]["lr_head"],
                      cfg["train"]["weight_decay"], freeze_backbone=False)
for g in opt.param_groups:
    n = sum(p.numel() for p in g["params"])
    print(f"  lr={g['lr']:.1e}   params={n/1e6:6.2f}M")
del probe, opt

## 5.2  Smoke test — proves the pipeline is correct

2 folds, 1 epoch per phase, small images, a 300-image subset of `train`. This is
**not** a real result — it exists to catch bugs in minutes instead of hours.
`config.yaml` is never modified: every override here is a local copy.

In [ ]:
logs = []
def log(msg): logs.append(msg); print(msg)

smoke = cross_validate(
    train_df, cfg, device,
    n_folds=2, phase1_epochs=1, phase2_epochs=1,
    size_stage1=128, size_stage2=128, subset=300,
    log=log,
)
print("\nsmoke-test val QWK per fold:", [round(f["best_val_qwk"], 3) for f in smoke["folds"]])
print(f"mean +/- std: {smoke['val_qwk_mean']:.3f} +/- {smoke['val_qwk_std']:.3f}")
assert smoke["val_qwk_mean"] == smoke["val_qwk_mean"], "NaN in val QWK — investigate before the full run"
print("\nSANITY CHECK: PASS — pipeline is correct, safe to launch the full run.")

In [ ]:
# Plot the smoke-test curves for the first fold — same plotting code the full run uses.
h = pd.DataFrame([{"phase": r["phase"], "epoch_global": i,
                   "train_loss": r["train"]["loss"], "val_loss": r["val"]["loss"],
                   "train_qwk": r["train"]["qwk"], "val_qwk": r["val"]["qwk"]}
                  for i, r in enumerate(smoke["folds"][0]["history"])])
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].plot(h.epoch_global, h.train_loss, "o-", label="train"); axes[0].plot(h.epoch_global, h.val_loss, "o-", label="val")
axes[0].set_title("loss (smoke test, fold 0)"); axes[0].set_xlabel("epoch (phase1 -> phase2)"); axes[0].legend()
axes[1].plot(h.epoch_global, h.train_qwk, "o-", label="train"); axes[1].plot(h.epoch_global, h.val_qwk, "o-", label="val")
axes[1].set_title("quadratic weighted kappa"); axes[1].set_xlabel("epoch"); axes[1].legend()
plt.tight_layout(); plt.show()

## 5.3  The full run  *(GPU — Kaggle/Colab)*

Set `RUN_FULL_TRAINING = True` **only on a GPU runtime**. With the defaults in
`config.yaml` (3 folds x (5 + 20) epochs at 224 -> 380px) this takes roughly
20-40 minutes per fold on a Kaggle T4/P100 — a few hours total for all 3 folds.
On a laptop CPU/MPS the same call would take many hours; leave it `False` there
and treat §5.2 as the evidence the method works.

In [ ]:
RUN_FULL_TRAINING = False   # <-- set True on a GPU runtime to launch the real run

if RUN_FULL_TRAINING:
    full_logs = []
    def flog(msg): full_logs.append(msg); print(msg)
    full_summary = cross_validate(train_df, cfg, device, log=flog)   # no overrides = config.yaml defaults
    with open(Path(cfg["paths"]["outputs"]) / "metrics" / "train_log.txt", "w") as f:
        f.write("\n".join(full_logs))
    print(f"\nFull CV val QWK: {full_summary['val_qwk_mean']:.3f} +/- {full_summary['val_qwk_std']:.3f}")
else:
    print("RUN_FULL_TRAINING is False — skipping. Flip it to True on a GPU runtime "
          "(Kaggle/Colab) to produce the report-quality checkpoints and cv_summary.json.")

## 5.4  Reading back a completed full run

If `outputs/metrics/cv_summary.json` exists (from a full run, here or on Kaggle),
this cell reports the headline number and plots every fold's curves — the figures
notebook 06 will build on for the final evaluation.

In [ ]:
summary_path = Path(cfg["paths"]["outputs"]) / "metrics" / "cv_summary.json"
if summary_path.is_file():
    summary = json.loads(summary_path.read_text())
    print(f"folds: {summary['n_folds']} | val QWK = {summary['val_qwk_mean']:.3f} +/- {summary['val_qwk_std']:.3f}")
    fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
    for r in summary["folds"]:
        h = pd.DataFrame([{"i": i, "val_loss": e["val"]["loss"], "val_qwk": e["val"]["qwk"]}
                          for i, e in enumerate(r["history"])])
        axes[0].plot(h.i, h.val_loss, label=f"fold {r['fold']}")
        axes[1].plot(h.i, h.val_qwk, label=f"fold {r['fold']}")
    axes[0].set_title("validation loss"); axes[0].legend()
    axes[1].set_title("validation QWK");  axes[1].legend()
    plt.tight_layout()
    plt.savefig(Path(cfg["paths"]["figures_dir"]) / "05_cv_curves.png", dpi=150)
    plt.show()
else:
    print(f"No full run found yet at {summary_path}. Run §5.3 on a GPU, or copy its "
          f"output back into outputs/metrics/ once it finishes on Kaggle.")

## Notes for the report — reproducibility & leakage

* Every random source is seeded from `config.yaml: seed` (splits, fold assignment,
  weight init, augmentation, MixUp/CutMix draws).
* Each fold's **balancing (oversampling) and augmentation act only on that fold's
  training rows** — validation folds always use the plain resize+normalise
  transform, so no augmented or duplicated image ever contributes to a validation
  score.
* The held-out **test split is never touched in this notebook** — it is opened for
  the first and only time in `06_evaluate.ipynb`.
* Checkpointing on **val QWK** (not loss or accuracy) means the model saved is the
  one that agrees best with clinician grades in the order-aware sense — the metric
  that actually matters for a screening tool.

## Milestone status
- [x] M1-M4 (EDA, preprocessing, augmentation, model+losses)
- [x] **M5 — training strategy**: two-phase + discriminative LR + cosine restarts +
     progressive resizing + AMP + 3-fold CV + early stopping + TTA. Smoke-tested;
     full run pending a GPU pass.
- [ ] M6 — evaluation (curves, P/R/F1, QWK, ROC/PR, calibration, Grad-CAM) → `notebooks/06_evaluate.ipynb`